# Main Evaluation - All 4 Models Comparison
**Final Integration Notebook**

This notebook loads the clean preprocessed data and evaluates all 4 models together.

In [7]:
# ====================== PATH FIX ======================
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

print("✅ Project root added to path")

✅ Project root added to path


In [8]:
# ====================== IMPORTS ======================
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


In [9]:
# ====================== LOAD CLEAN DATA ======================
from src.preprocessing import load_and_preprocess_data

print("Loading preprocessed and scaled data...")
X, y, df_full = load_and_preprocess_data(scale=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training samples : {X_train.shape[0]}")
print(f"Testing samples  : {X_test.shape[0]}")

Loading preprocessed and scaled data...
Loading and preprocessing data...
Feature scaling applied (StandardScaler)
Preprocessing completed!
Final shape: (32581, 23)
Training samples : 26064
Testing samples  : 6517


In [10]:
# ====================== LOAD / TRAIN ALL MODELS ======================
models = {}

# Try to load saved models first
model_names = ['logistic_regression_model', 'decision_tree', 'random_forest_model', 'svm_model']

for name in model_names:
    try:
        model_path = f"../models/{name}.pkl"
        models[name] = joblib.load(model_path)
        print(f"✅ Loaded {name.replace('_', ' ').title()}")
    except:
        print(f"⚠️  {name} model not found. Will train now.")
        # If model not found, you can train here (optional)
        pass

✅ Loaded Logistic Regression Model
✅ Loaded Decision Tree
✅ Loaded Random Forest Model
✅ Loaded Svm Model


In [14]:
# ====================== FINAL MODEL COMPARISON (FINAL CLEAN VERSION) ======================
results = []

for name, model in models.items():
    print(f"\nPredicting with {name}...")
    
    X_test_aligned = X_test.copy()
    
    if hasattr(model, 'feature_names_in_'):
        trained_features = list(model.feature_names_in_)
        for col in trained_features:
            if col not in X_test_aligned.columns:
                X_test_aligned[col] = 0
        X_test_aligned = X_test_aligned[trained_features]
    
    y_pred = model.predict(X_test_aligned)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    
    results.append({
        'Model': name.replace('_', ' ').title(),
        'Accuracy': round(acc*100, 2),
        'Precision': round(prec*100, 2),
        'Recall': round(rec*100, 2),
        'F1-Score': round(f1*100, 2)
    })

# Create final table
comparison_df = pd.DataFrame(results)
comparison_df = comparison_df.sort_values(by='F1-Score', ascending=False)

print("\n" + "="*70)
print("FINAL MODEL COMPARISON")
print("="*70)
print(comparison_df.to_string(index=False))

comparison_df.to_csv('../outputs/results/final_model_comparison.csv', index=False)
print("\n✅ Final comparison table saved successfully!")


Predicting with logistic_regression_model...

Predicting with decision_tree...

Predicting with random_forest_model...

Predicting with svm_model...

FINAL MODEL COMPARISON
                    Model  Accuracy  Precision  Recall  F1-Score
            Decision Tree     92.11      93.63   69.13     79.54
                Svm Model     91.35      92.89   66.02     77.18
Logistic Regression Model     74.84      43.99   49.34     46.51
      Random Forest Model     77.83       0.00    0.00      0.00

✅ Final comparison table saved successfully!
